# بوصلة LM: نموذجك الخاص من الصفر
شغّل الخلايا بالترتيب. **أول شيء:** القائمة ‹وقت التشغيل› ← ‹تغيير نوع وقت التشغيل› ← اختر **GPU (T4)**.

> صدق: نموذج `small` (110 مليون) على T4 المجاني يحتاج **أياماً** (على عدة جلسات) ليرى مئات الملايين من التوكنات. النتيجة نموذج صغير يكمل الجمل ويجيب أسئلة بسيطة، **لا** يبني تطبيقات كاملة. التدريب يُحفظ في Google Drive ويُستأنف.

In [ ]:
!nvidia-smi | head -15
from google.colab import drive
drive.mount('/content/drive')

## 1) الكود (يُحفظ في Drive ليبقى بعد انتهاء الجلسة)

In [ ]:
import os
os.makedirs('/content/drive/MyDrive/bawsala-lm', exist_ok=True)
%cd /content/drive/MyDrive/bawsala-lm
if not os.path.exists('Karrar'):
    !git clone --depth 1 -b ccr-6b7e7377-qadw3u https://github.com/karrarkoqw4v-ux/Karrar.git
%cd Karrar/model/scratch
!pip -q install tokenizers datasets

## 2) جمع البيانات (نصوص مسموحة فقط)
يحمّل ويكيبيديا العربية (CC BY-SA). أضف نصوصك أنت في `data/raw/*.txt` (مقالات كتبتها، لهجتك...). **لا** تضف نصوصاً لا تملك حقّها.

In [ ]:
from datasets import load_dataset
import os
os.makedirs('data/raw/arwiki', exist_ok=True)
ds = load_dataset('wikimedia/wikipedia', '20231101.ar', split='train', streaming=True)
n = 0
with open('data/raw/arwiki/wiki.txt', 'w', encoding='utf-8') as f:
    for r in ds:
        f.write(r['text'].strip() + '\n\n'); n += 1
        if n >= 300000: break   # قلّل الرقم لتجربة أسرع
print(n, 'articles')

## 3) التوكنايزر والتجهيز

In [ ]:
!python tokenizer_train.py --input data/raw --vocab 32000
!python prep_data.py --input data/raw --target-tokens 600000000

## 4) التدريب الأولي (يُستأنف تلقائياً)
شغّل هذه الخلية مراراً: كل مرة تتوقف قبل انتهاء الجلسة (11 ساعة) وتحفظ، ثم تكمل من حيث توقفت بـ `--resume`. راقب `val_loss`: كلما نزل كان أفضل.

In [ ]:
!python train.py --preset small --max-steps 20000 --batch 4 --grad-accum 32 --lr 6e-4 --eval-every 250 --sample-every 1000 --time-limit 39000 --resume

## 5) تعليم المحادثة (الضبط)
اكتب حواراتك في `data/sft/mine/*.txt` بصيغة:
```
س: سؤالك
ج: الجواب الذي تريده
```
(سطر فارغ بين الحوارات). 30 مثالاً جاهزة فقط للشكل؛ المفيد يحتاج **مئات أو آلاف** من الأمثلة الجيدة.

In [ ]:
!python build_sft.py
!python sft.py --epochs 3 --batch 8

## 6) جرّب نموذجك

In [ ]:
!printf 'مرحبا\nما عاصمة العراق؟\n\n' | python chat.py

## 7) النتيجة
ملفاتك في Drive: `bawsala-lm/Karrar/model/scratch/out/` ← `chat.pt` (النموذج) و`tokenizer.json`. هذان هما نموذجك.
تشغيله داخل التطبيق على الجوال خطوة منفصلة (تحويل وتشغيل محلي)، لم أبنِها بعد.